In [5]:
import xarray as xr
import os
from glob import glob

# =====================================================
# CONFIG
# =====================================================
BASE_DIR = r"D:\Project Works\Drought_QLD\Data\SPEI"
OUTPUT_DIR = r"D:\Project Works\Drought_QLD\Data\Final DB for Project\SPEI"

os.makedirs(OUTPUT_DIR, exist_ok=True)

YEARS = range(2000, 2026)

# Variables expected in your SPI files
EXPECTED_VARIABLES = {
    "spei_01",
    "spei_03",
    "spei_06",
    "spei_12"
}

# =====================================================
# MAIN LOGIC
# =====================================================
for year in YEARS:
    print(f"\n🔄 Processing {year}")

    datasets = []

    # Find all NetCDF files containing the year
    nc_files = glob(
        os.path.join(BASE_DIR, "**", f"*{year}*.nc"),
        recursive=True
    )

    if not nc_files:
        print(f"⚠️ No files found for {year}")
        continue

    print(f"   Found {len(nc_files)} files")

    for nc_file in nc_files:
        filename = os.path.basename(nc_file)

        try:
            ds = xr.open_dataset(nc_file)

            # -------------------------------------------------
            # Find the variable from inside the NetCDF file
            # -------------------------------------------------
            available_variables = list(ds.data_vars)

            expected_found = [
                var for var in available_variables
                if var in EXPECTED_VARIABLES
            ]

            if expected_found:
                # Use recognised SPI variables
                variables_to_keep = expected_found

            elif len(available_variables) == 1:
                # If the file has only one data variable, use it
                variables_to_keep = available_variables

            else:
                print(
                    f"⚠️ Skipping {filename}: "
                    f"could not identify variable. "
                    f"Variables found: {available_variables}"
                )
                ds.close()
                continue

            selected_ds = ds[variables_to_keep].load()
            ds.close()

            datasets.append(selected_ds)

            print(
                f"   ✓ {filename}: "
                f"{', '.join(variables_to_keep)}"
            )

        except Exception as error:
            print(f"   ✗ Error reading {filename}: {error}")

    if not datasets:
        print(f"⚠️ No valid datasets available for {year}")
        continue

    # -------------------------------------------------
    # Merge all variables
    # -------------------------------------------------
    try:
        combined_ds = xr.merge(
            datasets,
            compat="override",
            join="outer"
        )

    except Exception as error:
        print(f"✗ Could not merge files for {year}: {error}")
        continue

    # -------------------------------------------------
    # Metadata
    # -------------------------------------------------
    combined_ds.attrs.update({
        "title": "Combined Queensland SPEI Dataset",
        "year": year,
        "source": "SILO rainfall data and calculated SPEI",
        "created_by": "xarray automated merge"
    })

    # -------------------------------------------------
    # Save NetCDF
    # -------------------------------------------------
    out_file = os.path.join(
        OUTPUT_DIR,
        f"QLD_SPEI_{year}.nc"
    )

    try:
        combined_ds.to_netcdf(out_file)
        combined_ds.close()

        print(f"✅ Saved: {out_file}")
        print(f"   Variables: {list(combined_ds.data_vars)}")

    except Exception as error:
        print(f"✗ Could not save {out_file}: {error}")

print("\n🎉 All available SPI variables combined successfully!")


🔄 Processing 2000
   Found 4 files
   ✓ SPEI_2000.nc: spei_gamma_01
   ✓ SPEI_2000.nc: spei_gamma_12
   ✓ SPEI_2000.nc: spei_gamma_03
   ✓ SPEI_2000.nc: spei_gamma_06
✅ Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\SPEI\QLD_SPEI_2000.nc
   Variables: ['spei_gamma_01', 'spei_gamma_12', 'spei_gamma_03', 'spei_gamma_06']

🔄 Processing 2001
   Found 4 files
   ✓ SPEI_2001.nc: spei_gamma_01
   ✓ SPEI_2001.nc: spei_gamma_12
   ✓ SPEI_2001.nc: spei_gamma_03
   ✓ SPEI_2001.nc: spei_gamma_06
✅ Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\SPEI\QLD_SPEI_2001.nc
   Variables: ['spei_gamma_01', 'spei_gamma_12', 'spei_gamma_03', 'spei_gamma_06']

🔄 Processing 2002
   Found 4 files
   ✓ SPEI_2002.nc: spei_gamma_01
   ✓ SPEI_2002.nc: spei_gamma_12
   ✓ SPEI_2002.nc: spei_gamma_03
   ✓ SPEI_2002.nc: spei_gamma_06
✅ Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\SPEI\QLD_SPEI_2002.nc
   Variables: ['spei_gamma_01', 'spei_gamma_12', 'spei_gamma_03', 'spei_

In [7]:
import xarray as xr
ds = xr.open_dataset(r'D:\Project Works\Drought_QLD\Data\Final DB for Project\Protracted Drought\QLD_SPI1_2000_Protracted_Drought.nc')
print(ds)

<xarray.Dataset> Size: 12MB
Dimensions:             (time: 12, lat: 382, lon: 311)
Coordinates:
  * time                (time) datetime64[ns] 96B 2000-01-16T12:00:00 ... 200...
  * lat                 (lat) float64 3kB -29.15 -29.1 -29.05 ... -10.15 -10.1
  * lon                 (lon) float64 2kB 138.0 138.1 138.1 ... 153.4 153.5
Data variables:
    spi_01              (time, lat, lon) float32 6MB ...
    protracted_drought  (time, lat, lon) float32 6MB ...
    bioregion_id        (lat, lon) float32 475kB ...
Attributes:
    protracted_drought_method:  SPI-1 drought threshold with bioregion-specif...
    recovery_periods_months:    {'BRB': 3, 'CYP': 3, 'CQC': 3, 'CHC': 5, 'DEU...
    year:                       2000
